In [ ]:
!pip install tqdm

import nltk as nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from tqdm import tqdm
from collections import defaultdict, Counter
import numpy as np
import math as math

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

stop_words = set(stopwords.words('english'))
df = pd.DataFrame(pd.read_json('/content/drive/MyDrive/Information_Retrieval/BM25/data/corpus.jsonl', lines=True))
df.drop(columns=['metadata'], inplace=True)
corpus_tokens = {}

def tokenize(text):
    tokens = word_tokenize(text.lower())
    filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words]
    return filtered_tokens

for index, row in tqdm(df.iterrows(), total=df.shape[0]):
    tokens = tokenize(row['text'])
    filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words]
    corpus_tokens[row['_id']] = filtered_tokens

100%|██████████| 171332/171332 [03:43<00:00, 768.13it/s]


In [ ]:
inverted_index = defaultdict(dict)
for doc_id, tokens in tqdm(corpus_tokens.items(), desc='Indexing...'):
    for term, frequency in Counter(tokens).items():
        inverted_index[term][doc_id] = frequency

Indexing...: 100%|██████████| 171332/171332 [00:18<00:00, 9185.87it/s]


In [ ]:
docs_len = {}
for index, row in tqdm(df.iterrows(), total=df.shape[0], desc='Calculating doc stats...'):
    docs_len[row['_id']] = len(corpus_tokens[row['_id']])

Calculating doc stats...: 100%|██████████| 171332/171332 [00:08<00:00, 19305.81it/s]


In [ ]:
N = len(df)
average_dl = sum(docs_len.values()) / N

def bm25_score(term, doc_id, k1=0.50, b=0.75):
  if term not in inverted_index or doc_id not in inverted_index[term]:
    return 0.0

  tf = inverted_index[term][doc_id]
  dl = docs_len[doc_id]
  df = len(inverted_index[term])
  idf = math.log((N - df + 0.5) / (df + 0.5))
  denom = tf + k1 * (1 - b + b * dl / average_dl)
  score = idf * (tf * (k1 + 1) / denom)
  return score


In [ ]:
query = 'what is the origin of COVID-19'
query_tokens = tokenize(query)
union_docs = set().union(*(inverted_index[t].keys() for t in query_tokens))

scores = defaultdict(float)
for doc_id in tqdm(union_docs, desc='Calculating scores...'):
    score = sum(bm25_score(t, doc_id) for t in query_tokens)
    scores[doc_id] = score

sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
sorted_scores = sorted_scores[:50]
sorted_scores

Calculating scores...: 100%|██████████| 2048/2048 [00:00<00:00, 214416.02it/s]


[('dv9m19yk', 6.200983725246245),
 ('vh96sjss', 6.153842788299772),
 ('e3wjo0yk', 5.884709587353263),
 ('8ccl9aui', 5.875222816461321),
 ('4uaa6kpg', 5.856872159613021),
 ('deajwhx0', 5.80852225796597),
 ('icwvm7jp', 5.80852225796597),
 ('d0x23frk', 5.782004562711643),
 ('ymhcouo5', 5.760964100630998),
 ('2vpvdm11', 5.757527943578666),
 ('021q9884', 5.757527943578666),
 ('wim5q9a5', 5.72646137032702),
 ('49360l2a', 5.724509430968271),
 ('v6ci69n0', 5.72125915410958),
 ('73ylxhb7', 5.714375471894268),
 ('l0kc731z', 5.673807299398842),
 ('9l97eihy', 5.67322448212839),
 ('cniyembt', 5.653545540143411),
 ('zd7smm8r', 5.643468831436701),
 ('ayj4z8qn', 5.623422793768791),
 ('us1spoxu', 5.617862018903611),
 ('z14rf85c', 5.586662437279868),
 ('jkhvcjcb', 5.583754935186628),
 ('fyrrwy9v', 5.583754935186628),
 ('hewbl5yu', 5.583754935186628),
 ('9t0bafyz', 5.564130132347043),
 ('7csfkoh8', 5.55548678615366),
 ('42wv7zl6', 5.545917026608841),
 ('2ntxpdke', 5.5410375313264515),
 ('0pbp97ik', 5.525